# 🎮 Game QoE Intelligence Dashboard (2026)
### Professional Analysis Suite for Game Quality of Experience
---
**Project Scope:** This notebook serves as the visualization and insight layer for the QoE Analysis engine. It transforms raw network captures and visual labels into actionable research data.

#### 📑 Table of Contents:
1. **[Setup](#setup):** Environment configuration and parameters.
2. **[Data Bridge](#bridge):** Dynamic loading of pre-processed backend results.
3. **[Ground Truth](#truth):** Visual QoE validation and distribution.
4. **[Network Discovery](#network):** Protocol signatures and traffic stability.
5. **[Feature Engineering](#features):** Label distribution and cleaning.
6. **[Correlation Analysis](#correlation):** Statistical relationships between network metrics.
7. **[Signature Visualization](#signatures):** Distribution and Density (KDE) analysis.

In [ ]:
from datetime import datetime
import time
notebook_start_time = time.time()
print(f"🚀 Overall Execution Started at: {datetime.now().strftime('%H:%M:%S')}")
print("-" * 40)
import os
import sys
from pathlib import Path
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
USE_PARALLEL = False
USE_PARALLEL_ML = True
CORES_TO_USE = max(1, multiprocessing.cpu_count() - 4)
INCLUDE_SVM = False
CURRENT_DIR = Path(os.getcwd())
BASE_DIR = CURRENT_DIR.parent
DATA_SUBFOLDER = "P++200"
RAW_DATA_PATH = BASE_DIR.parent / "adb" / "sessions_screencap_pc_roi" / DATA_SUBFOLDER
PRODUCTS_DIR = CURRENT_DIR / f"Products_{DATA_SUBFOLDER}"
os.makedirs(PRODUCTS_DIR, exist_ok=True)
GAME_PORT_RANGE = "10010-10025"
BPF_FILTER = f"udp and (portrange {GAME_PORT_RANGE})"
WINDOW_SIZE = 10
DECAY_RATE = 0.5
SAMPLE_DURATION = 1
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))
METADATA_FILENAME = "metadata_results_auto.csv"
NETWORK_ANALYSIS_FILENAME = "NetDiscovery_Michael_gm.xlsx"
FEATURES_FILENAME = "final_training_features.csv"
IMPORTANCE_FILENAME = "feature_importance_results.csv"
print(f"--- Environment Configured ---")
print(f"Working Directory: {CURRENT_DIR}")
print(f"Data Source: {RAW_DATA_PATH}")
print(f"Network Filter: {BPF_FILTER}")
print(f"Parallel Mode (I/O - Blocks 10,18): {USE_PARALLEL}")
print(f"Parallel Mode (ML - Block 34): {USE_PARALLEL_ML} (Using {CORES_TO_USE} cores)")


<a id="setup"></a>
## 🛠️ Step 1: Environment & Dependency Setup
In this section, we initialize the paths for **P++200** and import the specialized libraries for network analysis (`NFStreamer`) and machine learning (`scikit-learn`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import csv
import ast
import json
import uuid
from nfstream import NFStreamer
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from joblib import Parallel, delayed
from rank import rank_qoe_3_classes, rank_str_to_int_mapping_3_classes
from modelse import make_models
%matplotlib inline
sns.set_theme(style="whitegrid")
print("✅ All modules imported successfully.")


## 📋 Step 2: Model Feature Definition
Definition of the "Toolbox" — the full list of raw and statistical network features used for training our predictive models.

In [ ]:
FEATURE_COLUMNS = [
            'src2dst_avg_pkt_size',
            'src2dst_avg_pkt_size_mean',
            'src2dst_avg_pkt_size_median',
            'src2dst_avg_pkt_size_std',
            'src2dst_avg_pkt_size_min',
            'src2dst_avg_pkt_size_max',
            'src2dst_pps',
            'src2dst_pps_mean',
            'src2dst_pps_median',
            'src2dst_pps_std',
            'src2dst_pps_min',
            'src2dst_pps_max',
            'dst2src_avg_pkt_size',
            'dst2src_avg_pkt_size_mean',
            'dst2src_avg_pkt_size_median',
            'dst2src_avg_pkt_size_std',
            'dst2src_avg_pkt_size_min',
            'dst2src_avg_pkt_size_max',
            'dst2src_pps',
            'dst2src_pps_mean',
            'dst2src_pps_median',
            'dst2src_pps_std',
            'dst2src_pps_min',
            'dst2src_pps_max',
            'pkt_dir_ratio',
            'pkt_dir_ratio_mean',
            'pkt_dir_ratio_median',
            'pkt_dir_ratio_std',
            'pkt_dir_ratio_min',
            'pkt_dir_ratio_max'
]


<a id="bridge"></a>
## 🌉 Step 3: The Smart Data Loading Bridge
This critical block automatically detects the latest archive generated by the Backend engine. It ensures the notebook is always working with the most recent results without re-processing hours of data.

In [ ]:
print(f"📥 Loading analysis results from: {PRODUCTS_DIR}")
archive_files = list(PRODUCTS_DIR.glob("window_10_*.csv"))
if archive_files:
    latest_file = sorted(archive_files, key=lambda x: (x.stat().st_mtime, x.stat().st_size), reverse=True)[0]
    training_df = pd.read_csv(latest_file)
    df = training_df
    print(f"✅ Loaded latest features archive: {latest_file.name} ({len(df)} samples)")
else:
    default_features_path = PRODUCTS_DIR / FEATURES_FILENAME
    if default_features_path.exists():
        training_df = pd.read_csv(default_features_path)
        df = training_df
        print(f"✅ Loaded default features file: {FEATURES_FILENAME}")
    else:
        print(f"⚠️ Warning: {FEATURES_FILENAME} not found!")
metadata_path = PRODUCTS_DIR / METADATA_FILENAME
if metadata_path.exists():
    metadata = pd.read_csv(metadata_path)
    print(f"✅ Metadata loaded: {len(metadata)} runs.")
else:
    print(f"⚠️ Warning: {METADATA_FILENAME} not found!")
net_path = PRODUCTS_DIR / NETWORK_ANALYSIS_FILENAME
if net_path.exists():
    net_df = pd.read_excel(net_path)
    print(f"✅ Network Analysis loaded: {len(net_df)} sessions.")
else:
    print(f"⚠️ Warning: {NETWORK_ANALYSIS_FILENAME} not found!")
imp_path = PRODUCTS_DIR / IMPORTANCE_FILENAME
if imp_path.exists():
    imp_df = pd.read_csv(imp_path)
    print(f"✅ Feature Importance loaded.")
else:
    print(f"⚠️ Warning: {IMPORTANCE_FILENAME} not found!")


<a id="truth"></a>
## 👁️ Step 4: Ground Truth & Visual Validation
Synchronizing the visual data (ping from screenshots) with the timeline to ensure our labels are accurate before analysis.

In [ ]:
def hhmmss_to_seconds(hhmmss: int) -> int:
    h = hhmmss // 10000
    m = (hhmmss // 100) % 100
    s = hhmmss % 100
    return h * 3600 + m * 60 + s
def to_qoe_class_zero(ping):
    if pd.isna(ping): return 0
    try:
        ping = float(ping)
    except Exception:
        return 0
    label = rank_qoe_3_classes(ping)
    return rank_str_to_int_mapping_3_classes.get(label, 0)
def build_series_from_row(row):
    start_hhmmss = int(row["folder"].split("_")[-1])
    start_sec = hhmmss_to_seconds(start_hhmmss)
    pairs = ast.literal_eval(row["time_value_pairs"])
    xs, ys = [], []
    for t_str, ping in pairs.items():
        dt = hhmmss_to_seconds(int(t_str)) - start_sec
        if dt < 0: continue
        xs.append(dt)
        ys.append(to_qoe_class_zero(ping))
    return pd.DataFrame({"t_sec": xs, "qoe": ys}).sort_values("t_sec")


In [ ]:
row0 = metadata.iloc[12]
s0 = build_series_from_row(row0)
mask0 = (s0["qoe"] == 0)
plt.figure(figsize=(12,4))
qoe_line = s0["qoe"].where(~mask0, np.nan)
plt.plot(s0["t_sec"], qoe_line, marker="o", linestyle="-", label="qoe (1/2/3)")
plt.scatter(s0.loc[mask0, "t_sec"], s0.loc[mask0, "qoe"],
            color="red", zorder=5, label="qoe=0")
plt.yticks([0,1,2,3])
plt.ylim(-0.5, 3.5)
plt.xlabel("Time (seconds from run start)")
plt.ylabel("QoE class (0/1/2/3)")
plt.title(f"QoE over time - {row0['folder']} / {row0['pcapng_file']}")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
def sume_ech_class(row):
    folder = row["folder"]
    pairs = ast.literal_eval(row["time_value_pairs"])
    class_1, class_2, class_3 = [], [], []
    for _t_str, ping in pairs.items():
        if ping is None:
            continue
        q = to_qoe_class_zero(ping)
        if q == 1: class_1.append(ping)
        elif q == 2: class_2.append(ping)
        elif q == 3: class_3.append(ping)
    return class_1, class_2, class_3
all_c1, all_c2, all_c3 = [], [], []
for i in metadata.index:
    row = metadata.loc[i]
    c1, c2, c3 = sume_ech_class(row)
    all_c1.append(len(c1))
    all_c2.append(len(c2))
    all_c3.append(len(c3))
print(f"Class 1 count: {sum(all_c1)}, Class 2 count: {sum(all_c2)}, Class 3 count: {sum(all_c3)}")
print(f"Class 1 std: {np.std(all_c1)}, Class 2 std: {np.std(all_c2)}, Class 3 std: {np.std(all_c3)}")


<a id="network"></a>
## 🌐 Step 5: Network Discovery & Protocol Validation
Quantitative proof that the captured traffic is game-related (UDP dominance) and validation of connection density across 302 sessions.

In [ ]:
udp_avg = net_df['udp_pct'].mean()
tcp_avg = net_df['tcp_pct'].mean()
plt.figure(figsize=(8, 6))
colors = ['#4CAF50', '#FFC107']
plt.pie([udp_avg, tcp_avg], labels=['UDP (Game Traffic)', 'TCP'],
        autopct='%1.1f%%', startangle=140, explode=(0.1, 0), shadow=True)
plt.title('Overall Network Protocol Distribution\n(Validation of Game Traffic Capture)')
plt.show()
print(f"Total sessions analyzed: {len(net_df)}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
ax1.hist(net_df['avg_iat_ms'], bins=20, color='skyblue', edgecolor='black', alpha=0.7)
ax1.axvline(net_df['avg_iat_ms'].median(), color='red', linestyle='dashed', linewidth=1.5, label='Median IAT')
ax1.set_title('Network Jitter Distribution\n(Average IAT per Session)')
ax1.set_xlabel('Mean Inter-Arrival Time (ms)')
ax1.set_ylabel('Frequency (Number of Sessions)')
ax1.legend()
ax2.scatter(net_df.index, net_df['avg_pkt_size'], alpha=0.6, color='purple', s=40)
ax2.set_title('Mean Packet Size per Session\n(Consistency Validation)')
ax2.set_xlabel('Session Index')
ax2.set_ylabel('Average Packet Size (Bytes)')
ax2.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
xlsx_name = NETWORK_ANALYSIS_FILENAME
if 'net_df' in locals():
    plot_df = net_df
elif (PRODUCTS_DIR / xlsx_name).exists():
    plot_df = pd.read_excel(PRODUCTS_DIR / xlsx_name)
else:
    plot_df = None
    print(f"Error: Could not find '{xlsx_name}' in the folder. Please ensure the file exists.")
if plot_df is not None:
    plt.figure(figsize=(12, 6))
    plt.scatter(plot_df.index, plot_df['total_flows'], alpha=0.6, color='dodgerblue', s=50, edgecolors='k')
    avg_flows = plot_df['total_flows'].mean()
    plt.axhline(avg_flows, color='red', linestyle='--', linewidth=2, label=f'Average Flows: {avg_flows:.1f}')
    plt.title('Network Flow Density per Session\n(Connectivity Profile Validation)', fontsize=14, pad=15)
    plt.xlabel('Session Index', fontsize=12)
    plt.ylabel('Number of Unique Flows', fontsize=12)
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()


<a id="features"></a>
## 🧹 Step 6: Feature Cleaning & Label Distribution
Preparing the dataset for Machine Learning by removing incomplete samples and calculating the target class distribution.

In [ ]:
df = training_df
df = df.dropna(subset=['ping'])
df = df.dropna(subset=FEATURE_COLUMNS)
df['y'] = df['ping'].apply(lambda p: rank_str_to_int_mapping_3_classes.get(rank_qoe_3_classes(p), 0))
df = df[df['y'] > 0].copy()
counts = df['y'].value_counts()
plt.figure(figsize=(6,6))
plt.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Distribution by Label')
plt.axis('equal')
plt.show()


<a id="correlation"></a>
## 📊 Step 7: Statistical Correlation Insights
Using **Spearman Correlation** to identify redundant features and discover which network metrics are the strongest "signals" for high-latency events.

In [ ]:
corr = df[FEATURE_COLUMNS].corr(method='spearman')
plt.figure(figsize=(14, 12))
sns.heatmap(
    corr,
    cmap='coolwarm',
    center=0,
    square=True,
    cbar_kws={'shrink': 0.8}
)
plt.title("Spearman Correlation Between Features")
plt.show()


In [ ]:
corr_features = df[FEATURE_COLUMNS].corr(method='spearman')
threshold = 0.85
high_corr = []
for i in range(len(corr_features.columns)):
    for j in range(i + 1, len(corr_features.columns)):
        val = corr_features.iloc[i, j]
        if abs(val) > threshold:
            high_corr.append((
                corr_features.columns[i],
                corr_features.columns[j],
                val
            ))
high_corr_df = pd.DataFrame(high_corr, columns=['feature_1', 'feature_2', 'spearman_corr'])
high_corr_df = high_corr_df.sort_values(by='spearman_corr', key=np.abs, ascending=False)
print(high_corr_df.head(20))


In [ ]:
for _, row in high_corr_df.head(5).iterrows():
    a, b = row['feature_1'], row['feature_2']
    plt.scatter(df[a], df[b], alpha=0.3)
    plt.xlabel(a)
    plt.ylabel(b)
    plt.title(f"Spearman={row['spearman_corr']:.2f}")
    plt.show()


In [ ]:
print("\n=== Correlation by classes ===")
df = df.dropna(subset=['ping'])
df['qoe_score'] = df['ping'].apply(lambda p: rank_str_to_int_mapping_3_classes.get(rank_qoe_3_classes(p), 0))
for label in sorted(df['qoe_score'].unique()):
    sub = df[df['qoe_score'] == label]
    corr = sub[FEATURE_COLUMNS].corr(method='spearman')
    print(f"\n=== Class {label} ===")
    print(corr.mean().sort_values(ascending=False).head(5))


In [ ]:
df.corr(method='spearman', numeric_only=True)['qoe_score'].sort_values(ascending=False)


In [ ]:
per_class_top = {}
for cls in sorted(df['qoe_score'].unique()):
    sub = df[df['qoe_score'] == cls]
    corr = sub[FEATURE_COLUMNS].corr(method='spearman', numeric_only=True)
    per_class_top[cls] = corr.abs().mean()
df_compare = pd.DataFrame(per_class_top)
df_compare['range'] = df_compare.max(axis=1) - df_compare.min(axis=1)
df_compare.sort_values('range', ascending=False).head(10)


<a id="signatures"></a>
## 📈 Step 8: Network Signatures (Distribution & KDE)
Final visualization of how each network feature separates the QoE classes. Distinct curves indicate high predictive power.

In [ ]:
for col in FEATURE_COLUMNS:
    plt.figure()
    data_all = df[col].replace([np.inf, -np.inf], np.nan).dropna()
    plt.hist(data_all, bins=40, alpha=0.3, label="All QoE", density=True)
    for cls in sorted(df['qoe_score'].unique()):
        data = df.loc[df['qoe_score'] == cls, col].replace([np.inf, -np.inf], np.nan).dropna()
        plt.hist(data, bins=40, alpha=0.5, label=f"QoE {cls}", density=True)
    plt.title(col)
    plt.legend()
    plt.show()


In [ ]:
for col in FEATURE_COLUMNS:
    plt.figure()
    for cls in sorted(df['qoe_score'].unique()):
        sns.kdeplot(
            df[df['qoe_score'] == cls][col].dropna(),
            label=f"QoE {cls}",
            fill=True
        )
    plt.title(col)
    plt.legend()
    plt.show()


In [ ]:
notebook_end_time = time.time()
total_duration = notebook_end_time - notebook_start_time
current_time = datetime.now().strftime('%H:%M:%S')
print(f"🏁 Notebook Visualization Completed in: {total_duration/60:.2f} minutes at {current_time}")


## 💾 Step 9: Final Persistence
Exporting the processed feature importance results for comparative study and archiving.

In [ ]:
imp_df.to_csv(PRODUCTS_DIR / IMPORTANCE_FILENAME , index=False)
